# 01 - Feature Engineering

En este notebook preparamos los datos para el modelo. Partimos del dataset limpio que guardamos al final del EDA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('WineQT_clean.csv')
df.shape

In [ ]:
df.head()

## Transformaciones

En el EDA vimos que algunas variables tienen una distribución muy asimétrica (cola larga a la derecha). Aplicamos transformación logarítmica para reducir esa asimetría.

In [ ]:
# miramos la asimetría de cada variable para decidir cuales transformar
df.skew().sort_values(ascending=False)

In [ ]:
# las que tienen skew > 1 las transformamos
skewed_vars = df.skew()[df.skew() > 1].index.tolist()
skewed_vars

In [ ]:
# comparamos antes y despues para ver si mejora
fig, axes = plt.subplots(2, len(skewed_vars), figsize=(16, 5))

for i, col in enumerate(skewed_vars):
    axes[0, i].hist(df[col], bins=30, color='steelblue', edgecolor='white')
    axes[0, i].set_title(col + '\noriginal', fontsize=8)
    
    axes[1, i].hist(np.log1p(df[col]), bins=30, color='seagreen', edgecolor='white')
    axes[1, i].set_title(col + '\nlog1p', fontsize=8)

plt.tight_layout()
plt.show()

La transformación mejora bastante la simetría en la mayoría de variables. Usamos `log1p` en lugar de `log` porque algunas variables tienen valores 0 y `log(0)` daría error.

In [ ]:
df_fe = df.copy()

for col in skewed_vars:
    df_fe[col] = np.log1p(df_fe[col])

df_fe.head()

In [ ]:
# comprobamos que la asimetria ha bajado
print('Skew antes:')
print(df[skewed_vars].skew().round(2))
print()
print('Skew despues:')
print(df_fe[skewed_vars].skew().round(2))

## Separación train / test

Separamos antes de escalar para no usar información del test en el ajuste del scaler.

In [ ]:
X = df_fe.drop(columns=['quality'])
y = df_fe['quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test: {X_test.shape[0]} muestras')

In [ ]:
# comprobamos que quality tiene una distribucion parecida en train y test
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

y_train.value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('quality - train')
axes[0].tick_params(axis='x', rotation=0)

y_test.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('quality - test')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

La distribución de `quality` es similar en ambos conjuntos.

## Escalado

Las variables tienen escalas muy distintas, por ejemplo `alcohol` va de 8 a 15 y `total sulfur dioxide` de 6 a 289. La regresión lineal es sensible a esto, así que escalamos con StandardScaler para que todas tengan media 0 y desviación estándar 1.

El scaler se ajusta solo con los datos de train. Si lo ajustásemos con todo el dataset estaríamos pasando información del test al modelo.

In [ ]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled.describe()

In [ ]:
# medias cercanas a 0 y std cercanas a 1, todo correcto
print('Media train (debe ser ~0):')
print(X_train_scaled.mean().round(3))
print()
print('Std train (debe ser ~1):')
print(X_train_scaled.std().round(3))

## Guardamos los conjuntos para el modelado

In [ ]:
X_train_scaled.to_csv('X_train.csv', index=False)
X_test_scaled.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print('Guardados: X_train, X_test, y_train, y_test')